# NB6 (Colab) — test forecasts of the foundation-model rows A6a and A6b

D79, after the protocol freeze (frozen configuration `2d27ea1e82060a0f…`). Runs `tools/fm_infer.py` for the two frozen settings with their pinned weights revisions on the 17 test events, from `results/nb6_forecasts/fm_inputs/` (`python tools/nb6_forecast.py --fm-inputs`), and writes `results/nb6_forecasts/fm/<setting>.parquet` with a JSON record. Resumable: a setting whose output exists for the current inputs is skipped. No score is computed here.

**On Colab:** the Drive mirror (`python tools/make_drive_mirror.py --set nb6`) holds the notebook, `tools/fm_infer.py`, `src/pvcold/`, `requirements-fm.txt` and the inputs. Choose a GPU runtime, run all cells, end with `drive.flush_and_unmount()` (last cell) and Ctrl+S, then `python tools/fetch_drive_runs.py --set nb6`.

In [ ]:
import os, sys, json, hashlib, subprocess, time
from pathlib import Path
ON_COLAB = 'google.colab' in sys.modules
DRIVE_REPO = '/content/drive/MyDrive/pv-cold-start-nb6'
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT, PY, DEVICE = Path(DRIVE_REPO), sys.executable, 'cuda'
    subprocess.run([PY, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements-fm.txt')], check=True)
else:
    ROOT = Path.cwd().resolve(); ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
    PY = str(ROOT / '.venv_fm' / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')); DEVICE = 'cpu'
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
INP, RUNS = ROOT / 'results' / 'nb6_forecasts' / 'fm_inputs', ROOT / 'results' / 'nb6_forecasts' / 'fm'
man = json.loads((INP / 'inputs_manifest.json').read_text())
for f, h in man['sha256'].items():
    assert hashlib.sha256((INP / f).read_bytes()).hexdigest() == h, f'{f} differs from its manifest'
FROZEN_RUNS = [('A6a', 'chronos2', 1344, 'full', '29ec3766d36d6f73f0696f85560a422f50e8498c'), ('A6b', 'timesfm25', 1344, 'none', '1d952420fba87f3c6dee4f240de0f1a0fbc790e3')]   # (row, model, context, covariates, weights revision) from the frozen configuration
print('ROOT', ROOT, '| device', DEVICE, '| events', len(man['events']), '| tasks', man['n_tasks'])

In [ ]:
log = []
for row, model, L, cov, rev in FROZEN_RUNS:
    out = RUNS / f'{model}_c{L}_{cov}.parquet'
    js = out.with_suffix('.json')
    if out.exists() and js.exists() and json.loads(js.read_text())['inputs_sha256'] == man['sha256']:
        log.append({'row': row, 'setting': out.stem, 'status': 'done before'})
        continue
    t1 = time.time()
    r = subprocess.run([PY, str(ROOT / 'tools' / 'fm_infer.py'), '--model', model, '--context', str(L), '--cov', cov,
                        '--inputs', str(INP), '--out', str(out), '--device', DEVICE, '--revision', rev], cwd=str(ROOT))
    assert r.returncode == 0, f'{out.stem} failed'
    info = json.loads(js.read_text())
    assert info['revision'] == rev and info['revision_pinned'], 'weights revision differs from the frozen one'
    log.append({'row': row, 'setting': out.stem, 'status': 'run', 'runtime_s': round(time.time() - t1, 1),
                'n_rows': info['n_rows'], 'revision': info['revision']})
    print(log[-1], flush=True)
print(log)

In [ ]:
if ON_COLAB:
    drive.flush_and_unmount()